# 02 · Train & Evaluate — YOLOv8 on the façade class contract

**Module 4 · Unit 3 · Session 3**

Trains the detector, extracts the metrics table, saves the curves, and writes
everything into `results/` so the README's claims are generated rather than typed.

---

## Before you press Run all

**Runtime → Change runtime type → T4 GPU.** On CPU this still completes, but slowly.

### Which dataset will this use?

Controlled by `DATASET_SOURCE` in [`src/config.py`](../src/config.py):

| Value | What happens | Needs |
|---|---|---|
| `synthetic` *(default)* | Generates a procedural façade dataset locally | nothing |
| `roboflow` | Downloads your own Roboflow project | API key + workspace/project in config |
| `universe` | Downloads a forked Roboflow Universe project | same |

The default is `synthetic` so that **this notebook runs end to end for a stranger with
no account**. That is the reproducibility requirement, taken literally.

> ⚠️ **Synthetic results are not a performance claim.** They prove the pipeline
> executes. Everything produced from them is stamped `VERIFICATION RUN`. Point
> `DATASET_SOURCE` at a real dataset before quoting a single number as a result.


In [ ]:
# ── CELL 1 ── Environment bootstrap. Run this first, every time.
# Works in Colab (clones the repo) and locally (uses the repo you are in).
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/OmarEAbdelaal/ZIGURAT-AI-AECO-Masters_Group-2.git"
SUBDIR   = "m4u3-facade-ve-vision"   # set to "" if the project is at the repo root

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    root = pathlib.Path('/content') / pathlib.Path(REPO_URL).stem
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    PROJECT = root / SUBDIR if SUBDIR else root
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT / 'requirements.txt')], check=True)
else:
    here = pathlib.Path.cwd()
    PROJECT = next((p for p in [here, *here.parents] if (p / 'src' / 'config.py').exists()), here)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
print('project root :', PROJECT)

import config as C
print()
print(C.summary())


## 1 · Roboflow credentials (skip if using the synthetic default)

The API key is read from a **Colab secret**, never typed into a cell and never
committed. Sidebar → 🔑 → new secret named `ROBOFLOW_API_KEY` → enable for this
notebook.


In [ ]:
# Uncomment and edit to train on your own Roboflow dataset:
#
# C.DATASET_SOURCE     = 'roboflow'
# C.ROBOFLOW_WORKSPACE = 'your-workspace'
# C.ROBOFLOW_PROJECT   = 'your-project'
# C.ROBOFLOW_VERSION   = 1

print('dataset source :', C.DATASET_SOURCE)
print('api key found  :', bool(C.api_key()))


## 2 · Resolve the dataset and run a health check

Session 2's three pillars: quality, structure (the split), consistency. The numbers
below evidence the second and expose the class imbalance that drives the error
analysis later.


In [ ]:
import json
from dataset import resolve_dataset, describe

data_yaml = resolve_dataset()
health = describe(data_yaml)
print(json.dumps(health, indent=2))

print()
print('split      :', health['split_ratio'])
print('imbalance  :', health['imbalance_ratio'], ': 1  (most vs least frequent class)')
assert health['splits']['train']['images'] > 0, 'no training images resolved'


## 3 · Look at the data before training on it

GIGO. Five minutes here saves an afternoon of debugging a model that learned your
labelling mistakes faithfully.


In [ ]:
import matplotlib.pyplot as plt, yaml
from PIL import Image
from pathlib import Path
from error_analysis import _load_gt

cfg  = yaml.safe_load(Path(data_yaml).read_text())
root = Path(cfg.get('path', Path(data_yaml).parent))
idir = root / cfg['train']; ldir = idir.parent / 'labels'
samples = sorted(p for p in idir.iterdir() if p.suffix.lower() in ('.jpg','.png'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, p in zip(axes.ravel(), samples):
    im = Image.open(p).convert('RGB'); ax.imshow(im)
    for cid, (x0, y0, x1, y1) in _load_gt(ldir / f'{p.stem}.txt', *im.size):
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                                   edgecolor=['#1C60F3','#41CF97','#7B4FE0','#DD6B20','#17A2B8'][cid],
                                   linewidth=1.6))
    ax.set_title(p.name, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 4 · Train

Every parameter comes from `src/config.py`, so the README's reproducibility table and
the run cannot disagree. The training contract:

| Parameter | Value | Why |
|---|---|---|
| `model` | `yolov8n.pt` | Transfer learning from COCO. Nano first — scale up only if mAP plateaus rather than starting heavy |
| `epochs` | `30` | The brief's minimum. Session 3: under 30 underfits, over 100 memorises |
| `batch` | `16` | Session 3's balance point; fits a free T4 |
| `imgsz` | `640` | Standard. 1280 is for hairline cracks, not for windows |
| `seed` | `42` | Fixed, so a re-run reproduces |
| `patience` | `20` | Early stop — guards the validation-loss 'U-turn' |

**Watch `val/box_loss`.** If training loss falls while validation loss climbs, the
model has started memorising. `patience=20` stops it, but see it happen.


In [ ]:
from train import train

run_dir, metrics = train()
print('\nrun directory:', run_dir)


## 5 · The metrics table

Precision / Recall / mAP50 / mAP50-95, overall and per class.

**`mAP50-95` is this project's headline number, not `mAP50`.** The VE read-out is
computed from box *area*, so a box 20 % too large is a 20 % cost error that `mAP50`
scores as a clean hit. Only the IoU-strict metric sees it.
See [`docs/problem_framing.md`](../docs/problem_framing.md) §3.


In [ ]:
from IPython.display import Markdown, display
display(Markdown((C.METRICS_DIR / 'metrics.md').read_text()))


## 6 · The curves — the vitals check

Session 3's three scenarios: the **ski slope** (good), the **flatline** (bad learning
rate or bad labels), the **U-turn** (overfitting — stop).


In [ ]:
from IPython.display import Image as IPImage, display
for name in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    f = C.CURVES_DIR / name
    if f.exists():
        print(f'\n{name}')
        display(IPImage(filename=str(f), width=900))


## 7 · Where the weights went

`best.pt`, not `last.pt`. `last.pt` is wherever training happened to stop; `best.pt`
is the epoch that scored highest on validation.

**Publish it as a GitHub Release asset** and paste the URL into `WEIGHTS_URL` in
`src/config.py` — that is how a reader runs notebook `03` without training.


In [ ]:
w = C.RESULTS_DIR / 'weights' / 'best.pt'
print('weights :', w)
print('exists  :', w.exists())
print('size    :', f'{w.stat().st_size/1e6:.1f} MB' if w.exists() else '—')
print()
print('Next: notebook 03 builds the evidence pack and mines the error taxonomy.')
